In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms #
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# 1. 设置设备与超参数
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
learning_rate = 0.001
epochs = 10

# 2. 数据准备与增强 (针对图片数据)
# transforms 会把图片转为张量并进行标准化
transform = transforms.Compose([ #compose()把多个变换组合在一起
    transforms.ToTensor(), #ToTensor() [0,255]归一化到[0, 1]；维度转换 HWC -> CHW；Int -> Float
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
    # 逐通道归一化。第一个()是三通道的均值，第二个()是标准差，即归一化到 [-1, 1]
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
# transform=transform 会在加载数据时自动对每张图片进行上述变换
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

# 3. 定义 CNN 模型 (包含卷积、池化、BatchNorm 和 Dropout)
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 卷积块 1: 输入 3 通道 (RGB) -> 输出 32 通道
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), # 批归一化让模型变稳
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # (窗口大小,步长)。图片尺寸从 32x32 变为 16x16
        )
        # 输入形状：[Batch, 3, 32, 32]  输出形状：[Batch, 32, 16, 16]
        
        # 卷积块 2: 32 -> 64
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 16x16 -> 8x8
        )
        
        # 全连接层(线性层，nn.Linear)
        self.classifier = nn.Sequential( #nn.Linear 只接收一维向量y=wx+b
            nn.Flatten(),       # 自动拉直张量 [Batch, 64, 8, 8] -> [Batch, 64*8*8]
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.1),    # 丢弃法，每个神经元都有50%在这一轮被关，防止过拟合；通常放在全连接层（Linear）之后，激活函数之后
            nn.Linear(512, 10)  # 10个类别 不用再加softmax因为交叉熵损失计算时自带log_softmax
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.classifier(x)
        return x

model = MyCNN().to(device)

# 4. 定义损失函数和优化器
criterion = nn.CrossEntropyLoss() # 多分类标配
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
writer = SummaryWriter('runs/cnn_cifar10')

# 5. 训练循环
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # 标准五步法
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # 计算验证准确率
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images) #大小为[Batch_Size64, 10]
            _, predicted = torch.max(outputs.data, 1) #torch.max返回最大值的值与索引；1 代表在列的方向上寻找最大值（取一行的最大值）
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    
    # 记录到 TensorBoard
    writer.add_scalar('Loss/Train', avg_loss, epoch)
    writer.add_scalar('Accuracy/Test', acc, epoch)
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {acc:.2f}%")

writer.close()
torch.save(model.state_dict(), "cnn_cifar.pth")

Epoch [1/10], Loss: 1.2812, Accuracy: 63.75%
Epoch [2/10], Loss: 0.9386, Accuracy: 68.05%
Epoch [3/10], Loss: 0.8045, Accuracy: 69.70%
Epoch [4/10], Loss: 0.7068, Accuracy: 71.96%
Epoch [5/10], Loss: 0.6227, Accuracy: 71.80%
Epoch [6/10], Loss: 0.5502, Accuracy: 74.33%
Epoch [7/10], Loss: 0.4839, Accuracy: 74.54%
Epoch [8/10], Loss: 0.4114, Accuracy: 75.35%
Epoch [9/10], Loss: 0.3542, Accuracy: 75.34%
Epoch [10/10], Loss: 0.3102, Accuracy: 74.80%


In [ ]:
# 数据增强 防止过拟合，不是改变图片的分辨率，通过对原始图片进行各种“变形”，增加数据量 
# 水平翻转、随机裁剪、颜色抖动、随机旋转

# 上采样是指增大图像的分辨率（宽高）160x160 -> 320x320

# 迁移学习 是指在一个任务上训练好的模型，在另一个相关的任务上进行微调（fine-tuning）
# 解决 数据不足、算例有限问题
